### 导入包

In [1]:
import pandas as pd
import numpy as np
import sklearn
import plotly.express as px

### 读取数据

1.读取用户信息表、银行账单表以及信用卡账单表

In [ ]:
userinfo_df = pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\userinfo.csv")
userinfo_df.head(5)

2.读取逾期表，并将第一列删掉

In [ ]:
overdue_df = pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\overdue.csv")
overdue_df.drop(['Unnamed: 0'],axis=1,inplace=True)
overdue_df.columns=['new_user_id','违约','分数']
overdue_df

3.读取银行流水表

In [ ]:
bank_df = pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\bank.csv")
bank_df.head(5)

4.读取银行账单

In [ ]:
bill_df = pd.read_csv(r"D:\WeChat Files\wxid_vzlr2hkxrgil22\FileStorage\File\2024-09\bill.csv")
bill_df

### 离散变量编码

3.统计用户信息表中“职业”变量有几个属性，户口类型有几个属性，并将这些属性用顺序编码的方式来进行编码，婚姻，学历等同理

In [ ]:
## 将离散变量转换为字典
def todict(col):
    res={}
    cnt=0
    for e in col.unique():
        res[e]=cnt
        cnt+=1
    return res

In [ ]:
userinfo_df=userinfo_df.replace({'性别':{'女':0,'男':1,'未知':2}})

In [ ]:
marriage_map = {'未婚': 0, '已婚': 1, '离异': 2, '丧偶': 3,'其他情况':4}
hukou_map = {'农业户口': 0, '非农业户口': 1, '集体户口': 2, '家庭户口': 3, '其他户口': 4}

userinfo_df = userinfo_df.replace({'婚姻状态': marriage_map, '户口类型': hukou_map})

In [ ]:
dict1=todict(userinfo_df['教育程度'])
userinfo_df=userinfo_df.replace(dict1)

In [ ]:
userinfo_df["职业"] = userinfo_df["职业"].map({'公职':0,'企业':1,'事业单位':2,'无业':3,'学生':4})

In [ ]:
userinfo_df

### 数据三值

统计用户信息表中的缺失值，并打印输出

In [ ]:
userinfo_df.isna().sum()

### 数据合并

In [ ]:
occupation = userinfo_df[["new_user_id","职业"]]
default = overdue_df[["new_user_id","分数"]]
default1 = overdue_df[["new_user_id","违约"]]
fenshu_data = pd.merge(occupation, default, how="outer", on="new_user_id")
data = pd.merge(fenshu_data, default1, how="outer", on="new_user_id")
data

# 相关系数

 $r=\frac{\sum_{i=1}^{n}(x_{i}-\overline{x})(y_{i}-\overline{y})}{\sqrt{\sum_{i=1}^{n}{(x_{i}-\overline{x})}^{2}}\sqrt{\sum_{i=1}^{n}{(y_{i}-\overline{y})}^{2}}}$

2.计算两组数据的相关系数，相关系数请自行编程实现，不要使用corr函数。并将结果与corr函数的结果对比。

In [ ]:
x, y = merge['工资平均收入'], merge['分数']
meanx = merge['工资平均收入'].mean()
meany = merge['分数'].mean()

NameError: name 'merge' is not defined

In [ ]:
upper = sum( (x - meanx) * (y - meany) )
lower = (sum((x - meanx) ** 2) ** 0.5) * (sum((y - meany) ** 2) ** 0.5)
r = upper / lower
print(f"相关系数：{r}") #使用“{}”作为占位符，将变量的值插入到字符串中

相关系数：0.13633187791178358


In [ ]:
x.corr(y)

0.13633187791178372

# 线性回归方法一：解方程

$\theta_{0}=\frac{\sum_{i=1}^{n}(x_{i})^{2}\sum_{i=1}^{n}y_{i}-\sum_{i=1}^{n}x_{i}\sum_{i=1}^{n}x_{i}y_{i}}{n\sum_{i=1}^{n}(x_{i})^{2}-(\sum_{i=1}^{n}x_{i})^{2}}$

$\theta_{1}=\frac{n\sum_{i=1}^{n}x_{i}y_{i}-\sum_{i=1}^{n}x_{i}\sum_{i=1}^{n}y_{i}}{n\sum_{i=1}^{n}(x_{i})^{2}-(\sum_{i=1}^{n}x_{i})^{2}}$

3.按照给出的线性回归流程，使用解方程的方法编程计算𝜃_0和𝜃_1

In [ ]:
xs1 = sum(x)
xs2 = sum(x ** 2)
ys1 = sum(y)
ys2 = sum(y ** 2)
xy  = sum(x * y)
print(f"xs1 = {xs1}; xs2 = {xs2};\nys1 = {ys1}; ys2 = {ys2};\nxy = {xy};")

xs1 = 25189.996409839492; xs2 = 1143956.5803103254;
ys1 = 45400.88523263889; ys2 = 3714755.8789129523;
xy = 2057534.710793386;


In [ ]:
n = len(x)
theta_0 = (xs2 * ys1 - xs1 * xy) / (n * xs2 - xs1 ** 2)
theta_1 = (n * xy - xs1 * ys1) / (n * xs2 - xs1 ** 2)
print(f'截距 theta_0：{theta_0}\n斜率 theta_1：{theta_1}')

截距 theta_0：71.37882475616765
斜率 theta_1：0.22684634706723292


# 线性回归方法二：网格搜索

4.使用网格搜索的方法编程计算𝜃_0和𝜃_1

我们进行如下设置：

斜率搜索范围都是  [−50,50]  ，步数  200  ，因此搜索步长自动计算为  (50−(−50))/200=0.5

截距搜索范围都是  [−50,50]  ，步数  200  ，因此搜索步长自动计算为  (50−(−50))/200=0.5

因此这里会计算  200×200=40000  次。不宜把步数设置太多，这将导致程序一直在执行。

 误差的方程是:$MSE=\sum_{i=1}^{n}(\theta_{0}+\theta_{1}x_{i}-y_{i})^{2}$

In [ ]:
xlim0, xlim1, xcnt = -50, 50, 200 
ylim0, ylim1, ycnt = -50, 50, 200 
xstep, ystep = (xlim1 - xlim0) / xcnt, (ylim1 - ylim0) / ycnt

"""
这样搜索出的值未必是最优的
你可以试试调整上面这些参数
比如设置：
xlim0, xlim1, xcnt = -50, 50, 100
ylim0, ylim1, ycnt = -50, 50, 100
就是在 [-50, -49, -48, ... , 49,50] 里搜索截距，在  [-50, -49, -48, ... , 49,50] 里搜索斜率

注意，计算量与 xcnt 、 ycnt 成正比，如果这两个变量设置得太大，你的程序可能卡死在这里。
"""

def cal_error(theta_0, theta_1):
    return sum((theta_0 + theta_1 * x - y) ** 2)

best_theta_0, best_theta_1, best_error = 0, 0, 2e9
from tqdm import tqdm
for t1 in tqdm([xlim0 + i * xstep for i in range(xcnt + 1)]):
    for t2 in [ylim0 + i * ystep for i in range(ycnt + 1)]:
        error = cal_error(t1, t2)
        if error < best_error:
            best_error = error
            best_theta_0 = t1
            best_theta_1 = t2

print(f"best_error: {best_error}, best_theta_0: {best_theta_0}, best_theta_1: {best_theta_1}")

100%|████████████████████████████████████████████████████████████████████████████████| 201/201 [00:17<00:00, 11.38it/s]

best_error: 8979.153572138313, best_theta_0: 36.5, best_theta_1: 1.0


### Machine Learning

In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import sklearn
from sklearn.linear_model import LinearRegression

#### Q1.利用梯度下降法求解$min_x(x-1)^2$的最优解

In [ ]:
def cost_function(x):
    return 2*(x-1)


In [ ]:
def gradient_descent(x0, alpha, N, epsilon):
    x=x0
    for i in range(N):
        grad=cost_function(x)
        x_new=x-alpha*grad
        if abs(x_new-x)<epsilon:
            break
        else:
            x=x_new
    return x

In [ ]:
gradient_descent(x0=0,alpha=0.05,N=200,epsilon=1e-6)

0.999990738612869

#### Q2.线性回归方法三：使用梯度下降法，寻找最优的参数值

* 一般的也被成为批量梯度下降法
* 随机梯度下降法:一次只处理一个样本，速度快，但可能不稳定
* 小批量梯度下降法:介于两者之间，速度比随机梯度下降法快，但比批量梯度下降法慢，一次处理一小批样本

#### Q3.在Python中，实现线性回归其实有现成的包：Sklearn。

#### Q4. Scikit-learn(Sklearn) 实现线性回归+最小二乘法